In [ ]:
!pip install -q -U google-genai gradio

from google import genai
from google.colab import userdata
import gradio as gr

# Google AI Studio API key
API_KEY = userdata.get("GEMMA_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMMA_API_KEY not found. "
        "Go to Colab Secrets and enable Notebook access."
    )

# Connect to Gemma
client = genai.Client(api_key=API_KEY)

MODEL = "gemma-4-31b-it"

print("GEMMA_API_KEY connected successfully!")
print("Gemma client initialized successfully!")
print(" Model:", MODEL)

GEMMA_API_KEY connected successfully!
Gemma client initialized successfully!
 Model: gemma-4-31b-it


In [ ]:
# ============================================================
#  GEMMA OUTDOOR PLAN GENERATOR
# ============================================================

def generate_outdoor_plan(activity, time_available, energy_level):

    if not activity.strip():
        return "⚠️ Please enter an outdoor activity."

    if not time_available.strip():
        return "⚠️ Please enter the available time."

    if not energy_level.strip():
        return "⚠️ Please select an energy level."

    prompt = f"""
You are "Ranchi EcoRoute", a simple outdoor activity planning assistant
for students in Ranchi, Jharkhand.

Your goal is to help the user spend less time on the screen and more time
doing a practical outdoor activity.

USER INPUT
Activity: {activity}
Time available: {time_available}
Energy level: {energy_level}

IMPORTANT RULES:
1. Do not claim real-time weather, live traffic, current opening hours,
   or live route conditions.
2. Do not give medical advice.
3. Keep the plan realistic for the user's available time and energy level.
4. Prefer simple outdoor activities that can be completed without complex
   equipment.
5. Mention that the user should independently verify weather, location
   access, and local conditions before leaving.
6. The screen should only be used briefly for planning; the actual goal
   is to go outdoors.

Return the answer in exactly this structure:

🌿 RANCHI ECOROUTE

🎯 Suggested Activity:
Give one clear activity.

📍 Suggested Place Type:
Suggest a suitable type of outdoor place in/around Ranchi.
Do not claim that a place is currently open or accessible.

⏱️ Suggested Plan:
Give a simple step-by-step plan that fits the available time.

🎒 What to Carry:
Give 3-5 practical items.

🛡️ Safety Tips:
Give 3 concise outdoor safety tips.

📵 Screen Break Goal:
Give one simple instruction for minimizing phone use
during the activity.

💡 Why This Plan:
Explain briefly why it fits the user's activity, time, and energy level.

⚠️ Reminder:
Verify weather, location access, and local conditions before heading out.
"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )

        return response.text.strip()

    except Exception as error:
        return f"❌ Gemma Error: {error}"


# ============================================================
# QUICK GEMMA TEST
# ============================================================

test_result = generate_outdoor_plan(
    "Morning Walk",
    "45 minutes",
    "Low"
)

print(test_result)

🌿 RANCHI ECOROUTE

🎯 Suggested Activity:
A Gentle Nature Stroll.

📍 Suggested Place Type:
A local city park or a quiet residential green belt in Ranchi.

⏱️ Suggested Plan:
1. Spend 5 minutes doing light stretching at home.
2. Take a slow, relaxed 15-minute walk to your chosen green space.
3. Spend 15 minutes walking at a leisurely pace, observing the trees and birds.
4. Use the final 10 minutes to walk slowly back home.

🎒 What to Carry:
- A bottle of water.
- Comfortable walking shoes.
- A small hand towel.
- Your phone (for emergency use only).

🛡️ Safety Tips:
- Stick to well-lit and populated walking paths.
- Maintain a steady, slow pace to avoid overexertion.
- Stay aware of your surroundings and other pedestrians.

📵 Screen Break Goal:
Keep your phone in your pocket or bag; do not check notifications until you return home.

💡 Why This Plan:
Since your energy is low and you have limited time, a slow-paced stroll in a green environment provides a mental refresh without physical st

In [16]:
# ============================================================
# RANCHI ECOROUTE GRADIO APP
# ============================================================

import gradio as gr


def plan_outdoor_activity(activity, time_available, energy_level):

    if not activity or not activity.strip():
        return "⚠️ Please enter an outdoor activity."

    if not time_available or not time_available.strip():
        return "⚠️ Please enter the time available."

    if not energy_level or not energy_level.strip():
        return "⚠️ Please select your energy level."

    return generate_outdoor_plan(
        activity.strip(),
        time_available.strip(),
        energy_level.strip()
    )


with gr.Blocks(
    title="Ranchi EcoRoute"
) as demo:

    gr.Markdown(
        """
# 🌿 RANCHI ECOROUTE

### Outdoor Activity Planner powered by Google Gemma

A small AI companion that helps students turn a few minutes of
screen time into a practical outdoor plan.

**Plan quickly. Go outside. Touch grass. 🌱**
"""
    )

    gr.Markdown(
        """
### How it works

Enter what you want to do, how much time you have, and your energy level.
Gemma creates a simple outdoor plan with practical safety guidance.

> ⚠️ This app does not provide live weather, traffic, opening hours,
> or real-time route information. Always verify local conditions yourself.
"""
    )

    with gr.Row():

        with gr.Column():

            activity_input = gr.Dropdown(
                choices=[
                    "Morning Walk",
                    "Nature Walk",
                    "Easy Hike",
                    "Nature Photography",
                    "Outdoor Reading",
                    "Jogging",
                    "Bird Watching",
                    "Other"
                ],
                value="Morning Walk",
                label="🌿 Outdoor Activity"
            )

            custom_activity = gr.Textbox(
                label="✏️ Custom Activity (optional)",
                placeholder="Example: Evening walk with a friend"
            )

            time_input = gr.Dropdown(
                choices=[
                    "30 minutes",
                    "45 minutes",
                    "60 minutes",
                    "90 minutes",
                    "2 hours"
                ],
                value="45 minutes",
                label="⏱️ Time Available"
            )

            energy_input = gr.Dropdown(
                choices=[
                    "Low",
                    "Medium",
                    "High"
                ],
                value="Low",
                label="⚡ Energy Level"
            )

            plan_button = gr.Button(
                "🌱 Create My Outdoor Plan",
                variant="primary"
            )

        with gr.Column():

            output = gr.Markdown(
                """
## 👋 Ready to go outside?

Choose an activity, set your available time and energy level,
then click **Create My Outdoor Plan**.
"""
            )

    def final_activity(activity, custom):
        if custom and custom.strip():
            return custom.strip()
        return activity

    plan_button.click(
        fn=lambda activity, custom, time, energy:
            plan_outdoor_activity(
                final_activity(activity, custom),
                time,
                energy
            ),
        inputs=[
            activity_input,
            custom_activity,
            time_input,
            energy_input
        ],
        outputs=output
    )

    gr.Markdown(
        """
---

### 🌱 Touch Grass Principle

The app is designed to keep planning short and encourage the user
to spend the majority of their time outside.

**Less screen time → more real-world activity.**

⚠️ Always verify weather, location access, local conditions, and
personal safety before heading outdoors.
"""
    )


demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ba233008d72ef9b1d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
